<a href="https://colab.research.google.com/github/olu1224/ztm-python-course-exercises/blob/main/PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install pyspark -q



In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Learning PySpark") \
    .master("local[*]") \
    .getOrCreate()

data = [
    ("Alice", "Engineering", 95000, 30),
    ("Bob", "Marketing", 72000, 25),
    ("Carol", "Engineering", 105000, 35),
    ("Dave", "Marketing", 68000, 28),
    ("Eve", "Engineering", 112000, 40),
    ("Frank", "HR", 58000, 33)
]

df = spark.createDataFrame(data, ["name", "department", "salary", "age"])
df.show()
df.printSchema()

+-----+-----------+------+---+
| name| department|salary|age|
+-----+-----------+------+---+
|Alice|Engineering| 95000| 30|
|  Bob|  Marketing| 72000| 25|
|Carol|Engineering|105000| 35|
| Dave|  Marketing| 68000| 28|
|  Eve|Engineering|112000| 40|
|Frank|         HR| 58000| 33|
+-----+-----------+------+---+

root
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- age: long (nullable = true)



In [5]:
# Select specific columns

df.select("name", "salary").show()

+-----+------+
| name|salary|
+-----+------+
|Alice| 95000|
|  Bob| 72000|
|Carol|105000|
| Dave| 68000|
|  Eve|112000|
|Frank| 58000|
+-----+------+



In [6]:
df.select("department", "age").show()

+-----------+---+
| department|age|
+-----------+---+
|Engineering| 30|
|  Marketing| 25|
|Engineering| 35|
|  Marketing| 28|
|Engineering| 40|
|         HR| 33|
+-----------+---+



In [7]:
#Filter rows where salary > 70000

df.filter(df.salary > 70000).show()

+-----+-----------+------+---+
| name| department|salary|age|
+-----+-----------+------+---+
|Alice|Engineering| 95000| 30|
|  Bob|  Marketing| 72000| 25|
|Carol|Engineering|105000| 35|
|  Eve|Engineering|112000| 40|
+-----+-----------+------+---+



In [8]:
df.filter(df.department == "Engineering").show()

+-----+-----------+------+---+
| name| department|salary|age|
+-----+-----------+------+---+
|Alice|Engineering| 95000| 30|
|Carol|Engineering|105000| 35|
|  Eve|Engineering|112000| 40|
+-----+-----------+------+---+



In [9]:
df.filter(df.age  <= 25).show()

+----+----------+------+---+
|name|department|salary|age|
+----+----------+------+---+
| Bob| Marketing| 72000| 25|
+----+----------+------+---+



In [10]:
df.filter(df.age  > 25).show()

+-----+-----------+------+---+
| name| department|salary|age|
+-----+-----------+------+---+
|Alice|Engineering| 95000| 30|
|Carol|Engineering|105000| 35|
| Dave|  Marketing| 68000| 28|
|  Eve|Engineering|112000| 40|
|Frank|         HR| 58000| 33|
+-----+-----------+------+---+



In [11]:
from pyspark.sql.functions import col

df.withColumn("Salary_after_tax", col("salary") * 0.8).show()

+-----+-----------+------+---+----------------+
| name| department|salary|age|Salary_after_tax|
+-----+-----------+------+---+----------------+
|Alice|Engineering| 95000| 30|         76000.0|
|  Bob|  Marketing| 72000| 25|         57600.0|
|Carol|Engineering|105000| 35|         84000.0|
| Dave|  Marketing| 68000| 28|         54400.0|
|  Eve|Engineering|112000| 40|         89600.0|
|Frank|         HR| 58000| 33|         46400.0|
+-----+-----------+------+---+----------------+



In [12]:
df.show()

+-----+-----------+------+---+
| name| department|salary|age|
+-----+-----------+------+---+
|Alice|Engineering| 95000| 30|
|  Bob|  Marketing| 72000| 25|
|Carol|Engineering|105000| 35|
| Dave|  Marketing| 68000| 28|
|  Eve|Engineering|112000| 40|
|Frank|         HR| 58000| 33|
+-----+-----------+------+---+



In [13]:
from pyspark.sql.functions import avg, max, count
df.groupBy("department").agg(
    count("*").alias("headcount"),
    avg("salary").alias("avg_salary"),
    max("salary").alias("max_salary")

).show()

+-----------+---------+----------+----------+
| department|headcount|avg_salary|max_salary|
+-----------+---------+----------+----------+
|Engineering|        3|  104000.0|    112000|
|  Marketing|        2|   70000.0|     72000|
|         HR|        1|   58000.0|     58000|
+-----------+---------+----------+----------+



In [14]:
from pyspark.sql.functions import min, sum, avg, max, count

df.groupBy("department").agg(
    min("salary").alias("min_salary"),
    sum("salary").alias("total_salary"),
    avg("salary").alias("avg_salary"),
    max("salary").alias("max_salary"),
    count("salary").alias("count_salary")

).show()


+-----------+----------+------------+----------+----------+------------+
| department|min_salary|total_salary|avg_salary|max_salary|count_salary|
+-----------+----------+------------+----------+----------+------------+
|Engineering|     95000|      312000|  104000.0|    112000|           3|
|  Marketing|     68000|      140000|   70000.0|     72000|           2|
|         HR|     58000|       58000|   58000.0|     58000|           1|
+-----------+----------+------------+----------+----------+------------+



In [15]:
# Register the DataFrame as a temporary SQL table | SQL queries

df.createOrReplaceTempView("employees")

# Now run raw SQL against it

result = spark.sql("""
    SELECT department, name, salary, avg(salary)
    FROM employees
    WHERE salary > 70000
    GROUP BY department, name, salary
    ORDER BY salary DESC
""")

result.show()


+-----------+-----+------+-----------+
| department| name|salary|avg(salary)|
+-----------+-----+------+-----------+
|Engineering|  Eve|112000|   112000.0|
|Engineering|Carol|105000|   105000.0|
|Engineering|Alice| 95000|    95000.0|
|  Marketing|  Bob| 72000|    72000.0|
+-----------+-----+------+-----------+

